In [11]:
%matplotlib inline


In [12]:
import json
import os

import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
from tqdm import tqdm

sns.set_context("notebook")
sns.set_style("whitegrid")

In [28]:
from inspect_ai.log import read_eval_log, ToolEvent

In [328]:
eval_log = read_eval_log("logs/ReAct_GPT5_paper-finder-test.eval")

In [359]:
import re
from collections import Counter, defaultdict

tool_calls = []
for sample in eval_log.samples:
    query = sample.metadata.get("raw_query")
    score_type = sample.metadata.get("score_type")
    if score_type == "specific_f1":
        print(sample.scores["score_paper_finder"].metadata)
    valid_corpusids_and_dates = sample.scores["score_paper_finder"].metadata[
        "valid_corpusids_and_dates_count"
    ]
    valid_corpusids_not_dates = sample.scores["score_paper_finder"].metadata[
        "valid_corpusids_not_dates_count"
    ]
    invalid_corpusids = sample.scores["score_paper_finder"].metadata[
        "invalid_corpusids_count"
    ]

    sample_tool_calls = []
    for event in sample.events:
        if isinstance(event, ToolEvent):
            sample_tool_calls.append(
                {
                    "function": event.function,
                    "arguments": event.arguments,
                    "result": event.result,
                    "error": event.error,
                }
            )
    tool_calls.append(
        {
            "query": query,
            "score_type": score_type,
            "calls": sample_tool_calls,
            "valid_corpusids_and_dates": valid_corpusids_and_dates,
            "valid_corpusids_not_dates": valid_corpusids_not_dates,
            "invalid_corpusids": invalid_corpusids,
        }
    )

funcs_used = []
funcs_ordered = []
tool_call_args = defaultdict(list)
for q in tool_calls:
    if "semantic" in q["score_type"]:
        print(f"Query: {q['query']}")
        print(f"Score Type: {q['score_type']}")
        
        query_funcs_used = []
        for call in q["calls"]:
            func_name = call["function"]
            if func_name == "snippet_search" and call["arguments"] and call["arguments"].get("paper_ids"):
                func_name += "_with_paper_ids"
            query_funcs_used.append(func_name)
            args = call["arguments"]
            args["__result"] = call["result"]
            args["__error"] = call["error"]
            args["__query"] = q["query"]
            tool_call_args[func_name].append(args)
        funcs_ordered.append(query_funcs_used)
        functions = Counter(query_funcs_used)
        funcs_used.append(functions)
        print(f"Functions: {functions}")

# create a dataframe from each tool call argument
arg_dfs = {function: pd.DataFrame(args) for function, args in tool_call_args.items()}
func_df = pd.DataFrame(funcs_used)
func_df.fillna(0).describe()

{'standard_f1': 1.0, 'relevant_predictions_at_full': 1, 'known_recall_at_full': 1.0, 'known_recall_at_estimate': 1.0, 'known_recall_at_30': 1.0, 'precision': 1.0, 'valid_corpusids_and_dates_count': 1, 'valid_corpusids_and_dates': ['13756489'], 'valid_corpusids_not_dates_count': 0, 'valid_corpusids_not_dates': [], 'invalid_corpusids_count': 0, 'invalid_corpusids': []}
{'standard_f1': 0.3333333333333333, 'relevant_predictions_at_full': 1, 'known_recall_at_full': 1.0, 'known_recall_at_estimate': 1.0, 'known_recall_at_30': 1.0, 'precision': 0.2, 'valid_corpusids_and_dates_count': 5, 'valid_corpusids_and_dates': ['215416146', '251597885', '253110666', '264418185', '239833492'], 'valid_corpusids_not_dates_count': 0, 'valid_corpusids_not_dates': [], 'invalid_corpusids_count': 0, 'invalid_corpusids': []}
{'standard_f1': 0.4, 'relevant_predictions_at_full': 1, 'known_recall_at_full': 1.0, 'known_recall_at_estimate': 1.0, 'known_recall_at_30': 1.0, 'precision': 0.25, 'valid_corpusids_and_dates_c

,search_papers_by_relevance,search_paper_by_title,submit,snippet_search_with_paper_ids,get_paper,snippet_search,get_paper_batch
count,194.000000,194.000000,194.0,194.000000,194.000000,194.000000,194.000000
mean,3.391753,1.546392,1.0,3.030928,0.355670,1.216495,0.092784
std,3.385604,3.311217,0.0,4.979655,1.102185,2.228998,0.340145
min,0.000000,0.000000,1.0,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,1.0,0.000000,0.000000,0.000000,0.000000
50%,3.000000,0.000000,1.0,0.000000,0.000000,0.000000,0.000000
75%,5.000000,1.000000,1.0,5.000000,0.000000,1.000000,0.000000
max,25.000000,21.000000,1.0,33.000000,8.000000,13.000000,2.000000


In [360]:
len([c for c in tool_calls if "semantic" in c["score_type"]])

194

### How many does it submit?

In [361]:
# how many valid / invalid corpusids are submitted
query_corpus_ids_submitted = []
for call in tool_calls:
    valid_corpusids_and_dates = call["valid_corpusids_and_dates"]
    valid_corpusids_not_dates = call["valid_corpusids_not_dates"]
    invalid_corpusids = call["invalid_corpusids"]

    query_corpus_ids_submitted.append(
        {
            "query": call["query"],
            "score_type": call["score_type"],
            "valid_corpusids_and_dates": valid_corpusids_and_dates,
            "valid_corpusids_not_dates": valid_corpusids_not_dates,
            "invalid_corpusids": invalid_corpusids,
        }
    )

submission_df = pd.DataFrame(query_corpus_ids_submitted)
submission_df.groupby("score_type").valid_corpusids_and_dates.describe().T

score_type,metadata_f1,semantic_f1,specific_f1
count,35.000000,194.000000,38.000000
mean,3.885714,8.438144,3.973684
std,3.305712,3.864115,2.745875
min,0.000000,1.000000,1.000000
25%,2.000000,6.000000,1.000000
50%,3.000000,9.000000,4.000000
75%,6.000000,10.000000,5.000000
max,12.000000,29.000000,11.000000


In [362]:
submission_df[submission_df.valid_corpusids_not_dates > 0].shape[0]

0

In [363]:
submission_df[submission_df.invalid_corpusids > 0]

,query,score_type,valid_corpusids_and_dates,valid_corpusids_not_dates,invalid_corpusids
25,CHI 2001 papers only by authors that only publ...,metadata_f1,0,0,1
31,papers citing the T5 paper and the spider pape...,metadata_f1,0,0,1


### What tools are used? 

In [367]:
func_df

,search_papers_by_relevance,search_paper_by_title,submit,snippet_search_with_paper_ids,get_paper,snippet_search,get_paper_batch
0,2.0,2.0,1,NaN,NaN,NaN,NaN
1,8.0,NaN,1,1.0,NaN,NaN,NaN
2,5.0,NaN,1,NaN,NaN,NaN,NaN
3,4.0,NaN,1,11.0,NaN,NaN,NaN
4,8.0,NaN,1,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...
189,5.0,NaN,1,NaN,NaN,NaN,NaN
190,NaN,NaN,1,NaN,NaN,1.0,NaN
191,5.0,NaN,1,NaN,NaN,NaN,NaN
192,7.0,5.0,1,4.0,NaN,NaN,NaN


In [369]:
func_df.describe()

,search_papers_by_relevance,search_paper_by_title,submit,snippet_search_with_paper_ids,get_paper,snippet_search,get_paper_batch
count,153.000000,66.000000,194.0,86.000000,28.000000,77.000000,15.000000
mean,4.300654,4.545455,1.0,6.837209,2.464286,3.064935,1.200000
std,3.258763,4.325886,0.0,5.474777,1.815206,2.622462,0.414039
min,1.000000,1.000000,1.0,1.000000,1.000000,1.000000,1.000000
25%,2.000000,1.000000,1.0,2.000000,1.000000,1.000000,1.000000
50%,4.000000,3.000000,1.0,6.000000,2.000000,2.000000,1.000000
75%,6.000000,6.000000,1.0,10.000000,3.250000,4.000000,1.000000
max,25.000000,21.000000,1.0,33.000000,8.000000,13.000000,2.000000


In [370]:
func_df.fillna(0).describe()

,search_papers_by_relevance,search_paper_by_title,submit,snippet_search_with_paper_ids,get_paper,snippet_search,get_paper_batch
count,194.000000,194.000000,194.0,194.000000,194.000000,194.000000,194.000000
mean,3.391753,1.546392,1.0,3.030928,0.355670,1.216495,0.092784
std,3.385604,3.311217,0.0,4.979655,1.102185,2.228998,0.340145
min,0.000000,0.000000,1.0,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,1.0,0.000000,0.000000,0.000000,0.000000
50%,3.000000,0.000000,1.0,0.000000,0.000000,0.000000,0.000000
75%,5.000000,1.000000,1.0,5.000000,0.000000,1.000000,0.000000
max,25.000000,21.000000,1.0,33.000000,8.000000,13.000000,2.000000


#### Tool combinations

In [368]:
fmelt = func_df.unstack().reset_index()
fmelt.columns = ["func", "query_id", "calls"]
fmelt = fmelt[(fmelt["calls"] > 0) & (fmelt.func!="submit")]
fmelt.groupby("query_id").apply(lambda x: sorted(x.func.unique())).value_counts()

/var/folders/pb/qz788cr500s2lnlvh1pmd3ph0000gn/T/ipykernel_9411/412874708.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  fmelt.groupby("query_id").apply(lambda x: sorted(x.func.unique())).value_counts()


[search_papers_by_relevance]                                                                                           29
[snippet_search]                                                                                                       29
[search_papers_by_relevance, snippet_search_with_paper_ids]                                                            29
[search_paper_by_title, search_papers_by_relevance, snippet_search_with_paper_ids]                                     18
[search_paper_by_title, search_papers_by_relevance]                                                                    14
[search_papers_by_relevance, snippet_search]                                                                           12
[get_paper, search_papers_by_relevance, snippet_search_with_paper_ids]                                                 10
[search_paper_by_title, search_papers_by_relevance, snippet_search]                                                     8
[get_paper, search_paper

In [371]:
fmelt.groupby("func").query_id.nunique().sort_values(ascending=False)

func
search_papers_by_relevance       153
snippet_search_with_paper_ids     86
snippet_search                    77
search_paper_by_title             66
get_paper                         28
get_paper_batch                   15
Name: query_id, dtype: int64

#### Tool chains

In [366]:
funcs_without_repeats = []
for funcs in funcs_ordered:
    funcs_wr = []
    current = None
    for func in funcs:
        if func != "submit" and func != current:
            funcs_wr.append(func)
            current = func
    funcs_without_repeats.append(tuple(funcs_wr))

Counter(funcs_without_repeats).most_common()


[(('search_papers_by_relevance',), 29),
 (('snippet_search',), 29),
 (('search_papers_by_relevance', 'snippet_search_with_paper_ids'), 26),
 (('search_papers_by_relevance', 'snippet_search_with_paper_ids', 'get_paper'),
  8),
 (('search_papers_by_relevance',
   'search_paper_by_title',
   'search_papers_by_relevance'),
  7),
 (('search_papers_by_relevance',
   'search_paper_by_title',
   'search_papers_by_relevance',
   'snippet_search_with_paper_ids'),
  6),
 (('search_papers_by_relevance', 'get_paper_batch'), 4),
 (('search_papers_by_relevance', 'snippet_search'), 4),
 (('search_papers_by_relevance',
   'snippet_search_with_paper_ids',
   'get_paper_batch'),
  3),
 (('search_papers_by_relevance',
   'search_paper_by_title',
   'snippet_search_with_paper_ids'),
  3),
 (('snippet_search', 'search_paper_by_title', 'snippet_search_with_paper_ids'),
  3),
 (('search_papers_by_relevance',
   'snippet_search',
   'search_papers_by_relevance',
   'snippet_search'),
  3),
 (('search_paper_by_

### Tool arguments

In [337]:
arg_dfs.keys()

dict_keys(['search_papers_by_relevance', 'search_paper_by_title', 'submit', 'snippet_search', 'get_paper', 'get_paper_batch'])

In [338]:
arg_dfs["snippet_search"].head()

,query,limit,paper_ids,__result,__error,__query
0,test-time adaptation machine translation evalu...,5.0,CorpusId:259370785,"[internal=None type='text' text='{\n ""data"": ...",None,Which papers explore the online adaptation of ...
1,compute-optimal scaling test-time compute can ...,5.0,CorpusId:271719990,"[internal=None type='text' text='{\n ""data"": ...",None,Many papers study optimal scaling laws for tra...
2,compute-optimal scaling 4x less test-time comp...,3.0,CorpusId:271719990,"[internal=None type='text' text='{\n ""data"": ...",None,Many papers study optimal scaling laws for tra...
3,coverage scales with the number of samples inf...,3.0,CorpusId:271571035,"[internal=None type='text' text='{\n ""data"": ...",None,Many papers study optimal scaling laws for tra...
4,compute-optimal inference smaller models Paret...,3.0,CorpusId:271601023,"[internal=None type='text' text='{\n ""data"": ...",None,Many papers study optimal scaling laws for tra...


In [339]:
snippet_searches_on_specific_id = arg_dfs["snippet_search"].paper_ids.notna().sum() / arg_dfs["snippet_search"].shape[0]
print(f'{snippet_searches_on_specific_id=:<.2f}')

snippet_searches_on_specific_id=0.71


In [340]:
arg_dfs["snippet_search"][arg_dfs["snippet_search"].paper_ids.notna()].limit.describe()

count    578.000000
mean       6.352941
std       10.094527
min        1.000000
25%        5.000000
50%        5.000000
75%        5.000000
max      200.000000
Name: limit, dtype: float64

In [341]:
arg_dfs["snippet_search"][arg_dfs["snippet_search"].paper_ids.isna()].limit.describe()

count    218.000000
mean      19.137615
std       11.470634
min        5.000000
25%       10.000000
50%       20.000000
75%       20.000000
max       50.000000
Name: limit, dtype: float64

In [342]:
arg_dfs["search_papers_by_relevance"].limit.describe()

count    622.000000
mean      19.726688
std        9.816639
min        5.000000
25%       20.000000
50%       20.000000
75%       20.000000
max       50.000000
Name: limit, dtype: float64

In [343]:
arg_dfs["search_papers_by_relevance"].fields.apply(lambda x: x.split(",") if pd.notna(x) else []).explode().value_counts()

fields
year                        447
venue                       447
title                       446
abstract                    446
corpusId                    446
authors                     438
citationCount               370
url                         366
influentialCitationCount    316
referenceCount              304
publicationDate              28
tldr                         28
urls                          4
Name: count, dtype: int64

In [344]:
arg_dfs["get_paper_batch"].fields.apply(lambda x: x.split(",") if pd.notna(x) else []).explode().value_counts()

fields
title                       14
abstract                    14
corpusId                    14
year                        11
venue                       11
url                         10
authors                      4
citationCount                2
referenceCount               2
influentialCitationCount     2
Name: count, dtype: int64

In [345]:
arg_dfs["search_paper_by_title"]["found"] = arg_dfs["search_paper_by_title"]["__error"].isna()

print("Successful title searches ratio:", arg_dfs["search_paper_by_title"]["found"].sum() / arg_dfs["search_paper_by_title"].shape[0])

Successful title searches ratio: 0.69


In [346]:
title_searches_by_query = arg_dfs["search_paper_by_title"].groupby(["__query", "found"]).size().unstack()
title_searches_by_query

found,False,True
__query,,
"AI2 dataset from around 2020 for reading comprehension metrics as opposed to models, acronym might be 'cocoa' or 'latte' or something. Possibly known for unusual punctuation splitting or not. Looking for a dataset with a name like 'LATTE' or something like that.",1.0,NaN
Are there any large-scale and open-source text simplification datasets dealing with long passages?,1.0,NaN
Are there any tools or studies that have focused on building a morphological analyzer specifically for handling multiple Arabic dialects?,1.0,1.0
Are there papers which propose a general data selection method based on information theory?,NaN,3.0
Can you point me to a paper that discussed transformer-based sentence embeddings?,1.0,7.0
...,...,...
papers about neural network activation functions,NaN,6.0
papers and datasets similar to DocBank or PubLayNet that include reading order,1.0,NaN
papers that use retrieval to improve large language models (LLMs) in-context by retrieving relevant documents and demonstrations,NaN,1.0


In [347]:
num_semantic_searches = len([c for c in tool_calls if "semantic" in c["score_type"]])
title_query_tries = title_searches_by_query.shape[0]
title_tries_query_ratio = title_query_tries / num_semantic_searches
title_query_successes = (title_searches_by_query[True] > 0).sum()
successful_title_searches_query_ratio = title_query_successes / num_semantic_searches
print(f"{title_tries_query_ratio=:<.2f} ({title_query_tries} / {num_semantic_searches})")
print(f"{successful_title_searches_query_ratio=:<.2f} ({title_query_successes} / {num_semantic_searches})")

title_tries_query_ratio=0.34 (66 / 194)
successful_title_searches_query_ratio=0.28 (54 / 194)
